# AIMO3 — Four-Phase Adaptation of a 120B Model for Tool-Augmented AIMO3 Reasoning
## Kaggle-Agnostic Inference Notebook

**Team:** Mohammad Shadab Alam (solo) · **Private LB:** X / X (TBD) · **Mean:** 38.56/50 (σ=1.67, n=35, best 42)

**Model:** /RecursiveRouge/gpt-oss-120b-merged-mxfp4-i700` (unmodified, no fine-tuning)
**Method:** 8 parallel tool-augmented attempts with majority voting (entropy plays a role) in case of a tie

**Requirements:** 1× A100 80GB or H100 80GB GPU runtime

This notebook is the Kaggle-agnostic version of `aimo_rag_baked_c283-v4`, created per the Writeup Prize reproducibility requirements. It pulls the base model from HuggingFace, contains the full inference pipeline, and runs on any machine with sufficient GPU memory.

**Links:**
- [Kaggle submission notebook](https://www.kaggle.com/code/outliar/aimo-rag-baked-c283?_gl=1*1ru74d1*_ga*MTEwOTc4Mjc5Ny4xNzU2NzQzMjE3*_ga_T7QHS60L4Q*czE3NzY2MDg5NDkkbzQzMiRnMSR0MTc3NjYxMDg4OCRqNTckbDAkaDA)
- [GitHub repo](https://github.com/codesrepo/aimo3-training-pipeline)
- [HuggingFace model](https://huggingface.co/RecursiveRouge/gpt-oss-120b-merged-mxfp4-i700/tree/main) (lora baked)


## 1. Install Dependencies

In [76]:
!pip install -q vllm openai openai_harmony transformers jupyter_client polars

## 2. Download Model

Downloads `/RecursiveRouge/gpt-oss-120b-merged-mxfp4-i700` from HuggingFace (~65GB). This is the model used in the competition  with LORA weights baked in.

In [77]:
import os

MODEL_DIR = "./gpt-oss-120b"

if not os.path.exists(MODEL_DIR):
    !hf download RecursiveRouge/gpt-oss-120b-merged-mxfp4-i700 --local-dir {MODEL_DIR}
    print(f"Model downloaded to {MODEL_DIR}")
else:
    print(f"Model already exists at {MODEL_DIR}")

Model already exists at ./gpt-oss-120b


## 3. Environment Setup

In [78]:
import os
import sys
import subprocess
import warnings

warnings.simplefilter('ignore')

os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True' # needed for google colab H100

print("Environment configured")

Environment configured


## 4. Imports

In [79]:
import gc
import re
import math
import time
import queue
import threading
import contextlib
import csv
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

from openai import OpenAI
from openai_harmony import (
    HarmonyEncodingName, load_harmony_encoding, SystemContent,
    ReasoningEffort, ToolNamespaceConfig, Author, Message,
    Role, TextContent, Conversation
)
from transformers import set_seed

print("All imports successful")

All imports successful


In [80]:
#import os, signal
#os.kill(os.getpid(), signal.SIGKILL)

## 5. Configuration

All parameters match the Kaggle submission exactly. The only change is `model_path` which points to the local HuggingFace download instead of the Kaggle dataset mount.

In [81]:
class CFG:

    system_prompt = (
        'You are a world-class IMO competitor. '
        'The final answer must be a non-negative integer between 0 and 99999. '
        'Place your final answer inside \\boxed{}.'
    )

    tool_prompt = (
        'Use this tool to execute Python code. '
        'The environment is a stateful Jupyter notebook. '
        'Always use print() to output results.'
    )

    preference_prompt = (
        'You have access to math, numpy, sympy, and mpmath.'
    )

    # Model — points to local HuggingFace download
    served_model_name = 'gpt-oss'
    model_path = './gpt-oss-120b'  # <-- Changed from Kaggle path

    # Sampling
    temperature = 1.0
    min_p = 0.02

    # Resource config
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    high_problem_timeout = 900
    base_problem_timeout = 300

    notebook_limit = 17520
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 6
    sandbox_timeout = 3

    stream_interval = 200
    context_tokens = 65536
    buffer_tokens = 512
    search_tokens = 32
    top_logprobs = 5
    batch_size = 32 #default 256
    early_stop = 4
    attempts = 8
    workers = 16
    turns = 128
    seed = 42

    use_rag = 0 # can be 0 or 1
    cert_path = "/kaggle/input/rag-traces/aimo_certs_283.jsonl"
    gpu_memory_utilization = 0.90 # default is 0.96
set_seed(CFG.seed)
print("Configuration loaded")

Configuration loaded


## 6. Harmony Template

In [82]:
class AIMO3Template:
    def __init__(self):
        pass

    def get_system_content(self, system_prompt, tool_config):
        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    def apply_chat_template(self, system_prompt, user_prompt, tool_config):
        system_content = self.get_system_content(system_prompt, tool_config)
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)
        user_message = Message.from_role_and_content(Role.USER, user_prompt)
        return [system_message, user_message]

## 7. Jupyter Sandbox

In [83]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None

        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import mpmath\n'
            'import itertools\n'
            'import collections\n'
            'mpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout

        msg_id = client.execute(
            code,
            store_history=True,
            allow_stdin=False,
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []

        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):

        self.execute(
            '%reset -f\n'
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import mpmath\n'
            'import itertools\n'
            'import collections\n'
            'mpmath.mp.dps = 64\n'
        )

    def __del__(self):

        self.close()

## 8. Python Tool

In [84]:
class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox

        self._owns_session = sandbox is None

        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:

        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    @property
    def instruction(self) -> str:

        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:

        return ToolNamespaceConfig(
            name='python',
            description=self.instruction,
            tools=[]
        )

    def _make_response(self, output: str, channel: str | None = None) -> Message:

        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')

        if channel:
            message = message.with_channel(channel)

        return message

    def process_sync_plus(self, message: Message) -> list[Message]:

        self._ensure_session()
        raw_script = message.content[0].text
        final_script = self._ensure_last_print(raw_script)

        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)

            except TimeoutError as exc:
                output = f'[ERROR] {exc}'

        return [self._make_response(output, channel=message.channel)]

## 9. Solver (Core Inference Engine)

This is the main solver class containing:
- `solve_problem()` — orchestrates 8 attempts with early stopping

In [85]:
import json
import os
import re
import random
import hashlib
from collections import defaultdict

# ---------------------------
# Preprocessing
# ---------------------------
_DIGITS_RE = re.compile(r"\d+")
_NON_WORD_RE = re.compile(r"[^\w\s]+", flags=re.UNICODE)  # remove punctuation/symbols (optional)

def preprocess_text(text: str) -> str:
    if text is None:
        return ""
    text = text.lower()

    # 1) remove digits first
    text = _DIGITS_RE.sub(" ", text)

    # 2) remove punctuation/symbols (keeps letters + underscore as "word chars")
    # text = _NON_WORD_RE.sub(" ", text)

    # 3) normalize whitespace
    return " ".join(text.split())

# ---------------------------
# Tokenization + shingles
# ---------------------------
_word_re = re.compile(r"[a-z0-9]+")

def tokenize(text: str):
    # consistent, simple tokenizer (ASCII letters/digits only)
    return _word_re.findall(preprocess_text(text))

def shingles(tokens, k=5):
    # word shingles (k-grams). For short docs, you can lower k to 3.
    if len(tokens) < k:
        return {" ".join(tokens)} if tokens else set()
    return {" ".join(tokens[i:i+k]) for i in range(len(tokens)-k+1)}

# ---------------------------
# MinHash (Jaccard estimator)
# ---------------------------
def _stable_hash(s: str) -> int:
    # stable across runs (unlike Python's built-in hash)
    return int(hashlib.md5(s.encode("utf-8")).hexdigest(), 16)

def make_minhash_funcs(n=128, seed=42):
    rng = random.Random(seed)
    # Use random linear hashes: (a*x + b) % prime
    # prime just above 2^32
    prime = 4294967311
    funcs = []
    for _ in range(n):
        a = rng.randrange(1, prime - 1)
        b = rng.randrange(0, prime - 1)
        funcs.append((a, b, prime))
    return funcs

def minhash_signature(shingle_set, funcs):
    # signature length = len(funcs)
    sig = []
    # pre-hash shingles to ints once
    xs = [_stable_hash(s) & 0xFFFFFFFF for s in shingle_set]
    for a, b, p in funcs:
        m = None
        for x in xs:
            hx = (a * x + b) % p
            if m is None or hx < m:
                m = hx
        sig.append(m if m is not None else 0)
    return tuple(sig)

# ---------------------------
# LSH (banding)
# ---------------------------
def lsh_buckets_from_sig(sig, bands=32):
    # Example: 128 sig -> 32 bands * 4 rows
    r = len(sig) // bands
    assert bands * r == len(sig)
    for i in range(bands):
        band = sig[i * r:(i + 1) * r]
        # band key: hash the band tuple
        key = hashlib.md5(repr((i, band)).encode("utf-8")).hexdigest()
        yield key

def jaccard(a_set, b_set):
    if not a_set and not b_set:
        return 1.0
    if not a_set or not b_set:
        return 0.0
    inter = len(a_set & b_set)
    union = len(a_set | b_set)
    return inter / union

# ---------------------------
# Index builder
# ---------------------------
class LSHMinHashIndex:
    def __init__(self, num_hashes=128, bands=32, shingle_k=5, seed=42):
        self.funcs = make_minhash_funcs(num_hashes, seed=seed)
        self.bands = bands
        self.shingle_k = shingle_k
        self.bucket = defaultdict(list)  # bucket_key -> [doc_id]
        self.doc_shingles = {}           # doc_id -> shingle_set
        self.doc_sig = {}               # doc_id -> signature

    def add(self, doc_id, text):
        toks = tokenize(text)
        sh = shingles(toks, k=self.shingle_k)
        sig = minhash_signature(sh, self.funcs)
        self.doc_shingles[doc_id] = sh
        self.doc_sig[doc_id] = sig
        for key in lsh_buckets_from_sig(sig, bands=self.bands):
            self.bucket[key].append(doc_id)

    def query(self, text, top_k=5, min_candidates=50):
        toks = tokenize(text)
        q_sh = shingles(toks, k=self.shingle_k)
        q_sig = minhash_signature(q_sh, self.funcs)

        # collect candidates
        cands = set()
        for key in lsh_buckets_from_sig(q_sig, bands=self.bands):
            for doc_id in self.bucket.get(key, []):
                cands.add(doc_id)

        # fallback if too few candidates
        if len(cands) < min_candidates:
            cands = set(self.doc_shingles.keys())

        scored = []
        for doc_id in cands:
            score = jaccard(q_sh, self.doc_shingles[doc_id])
            scored.append((score, doc_id))
        scored.sort(reverse=True)
        return scored[:top_k]

def build_cert_lsh_index(certs, num_hashes=128, bands=32, shingle_k=5, seed=42):
    idx = LSHMinHashIndex(num_hashes=num_hashes, bands=bands, shingle_k=shingle_k, seed=seed)
    cert_by_id = {}
    for i, cert in enumerate(certs):
        doc_id = cert.get("id", i)
        # ensure unique ids
        if doc_id in cert_by_id:
            doc_id = f"{doc_id}__{i}"
        cert["_lsh_id"] = doc_id
        cert_by_id[doc_id] = cert

        text = cert.get("problem_processed") or cert.get("problem") or ""
        idx.add(doc_id, text)
    return idx, cert_by_id

# ---------------------------
# CERT loader
# ---------------------------
def load_certs(_certs_path):
    # Load certs with embeddings (run after building aimo_certs_r4_with_embeddings.jsonl)
    CERT_EXAMPLES_WITH_EMBEDDINGS = []
    #_certs_path = "/kaggle/input/rag-traces/aimo_certs_219.jsonl"

    if os.path.exists(_certs_path):
        with open(_certs_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue

                cert = json.loads(line)

                # preprocess the "problem" field and store it as a new column/key
                problem_text = cert.get("problem", "")
                cert["problem_processed"] = preprocess_text(problem_text)

                CERT_EXAMPLES_WITH_EMBEDDINGS.append(cert)

        print(f"Loaded {len(CERT_EXAMPLES_WITH_EMBEDDINGS)} cert examples with embeddings from {_certs_path}")
    else:
        print(f"File not found: {_certs_path}. Run the previous cell to generate it.")

    return CERT_EXAMPLES_WITH_EMBEDDINGS

In [86]:
class AIMO3Solver:

    def __init__(self, cfg, port: int = 8000):

        self.cfg = cfg
        if self.cfg.use_rag:
            self.CERT_EXAMPLES_WITH_EMBEDDINGS = load_certs(self.cfg.cert_path)
            # Build fast MinHash-LSH index for cert retrieval (Jaccard over shingles)
            # This makes the "similar example hint" step fast even as certs grow.
            self.cert_lsh, self.cert_by_id = build_cert_lsh_index(
                self.CERT_EXAMPLES_WITH_EMBEDDINGS,
                num_hashes=128,
                bands=32,
                shingle_k=5,
                seed=self.cfg.seed if hasattr(self.cfg, "seed") else 42,
            )

        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()

        self._preload_model_weights()

        self.server_process = self._start_server()

        self.client = OpenAI(
            base_url=self.base_url,
            api_key=self.api_key,
            timeout=self.cfg.session_timeout
        )

        self._wait_for_server()
        self._initialize_kernels()

        self.notebook_start_time = time.time()
        self.problems_remaining = 50

    def _preload_model_weights(self) -> None:

        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()

        files_to_load = []
        total_size = 0

        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)

                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)

        def _read_file(path: str) -> None:

            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))

        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

    def _start_server(self) -> subprocess.Popen:

        cmd = [
            sys.executable,
            '-m',
            'vllm.entrypoints.openai.api_server',
            '--seed',
            str(self.cfg.seed),
            '--model',
            self.cfg.model_path,
            '--served-model-name',
            self.cfg.served_model_name,
            '--tensor-parallel-size',
            '1',
            '--max-num-seqs',
            str(self.cfg.batch_size),
            '--gpu-memory-utilization',
            str(self.cfg.gpu_memory_utilization),
            '--host',
            '0.0.0.0',
            '--port',
            str(self.port),
            '--dtype',
            self.cfg.dtype,
            '--kv-cache-dtype',
            self.cfg.kv_cache_dtype,
            #"--enable-lora",
            #"--max-lora-rank", str(LORA_RANK),
            #"--lora-modules", f"{LORA_NAME}={LORA_DIR}",
            '--max-model-len',
            str(self.cfg.context_tokens),
            '--stream-interval',
            str(self.cfg.stream_interval),
            '--async-scheduling',
            '--disable-log-stats',
            '--enable-prefix-caching'
        ]

        self.log_file = open('vllm_server.log', 'w')

        return subprocess.Popen(
            cmd,
            stdout=self.log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True
        )

    def _wait_for_server(self):

        print('Waiting for vLLM server...')
        start_time = time.time()

        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()

            if return_code is not None:
                self.log_file.flush()

                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()

                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')

            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')

                return

            except Exception:
                time.sleep(1)

        raise RuntimeError('Server failed to start (timeout).\n')

    def _initialize_kernels(self) -> None:

        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()

        self.sandbox_pool = queue.Queue()

        def _create_sandbox():

            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]

            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())

        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')

    def _scan_for_answer(self, text: str) -> int | None:

        pattern = r'\\boxed\s*\{\s*([0-9,]+)\s*\}'
        matches = re.findall(pattern, text)

        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)

                if 0 <= value <= 99999:
                    return value

            except ValueError:
                pass

        pattern = r'final\s+answer\s+is\s*([0-9,]+)'
        matches = re.findall(pattern, text, re.IGNORECASE)

        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)

                if 0 <= value <= 99999:
                    return value

            except ValueError:
                pass

        return None

    def _compute_mean_entropy(self, logprobs):
        if not logprobs: return float('inf')
        total, count = 0.0, 0
        for top_lp in logprobs:
            if isinstance(top_lp, dict) and top_lp:
                ent = sum(-math.exp(lp)*math.log2(math.exp(lp)) for lp in top_lp.values() if math.exp(lp) > 0)
                total += ent
                count += 1
        return total/count if count else float('inf')

    def _process_attempt(
        self,
        problem: str,
        system_prompt: str,
        attempt_index: int,
        stop_event: threading.Event,
        deadline: float
    ) -> dict:

        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1,
                'Answer': None,
                'Python Calls': 0,
                'Python Errors': 0,
                'Response Length': 0,
                'Entropy': float('inf')
            }

        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None

        logprobs_buffer = []

        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))

        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)

            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout,
                tool_prompt=self.cfg.tool_prompt,
                sandbox=sandbox
            )

            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt,
                problem,
                local_tool.tool_config
            )

            conversation = Conversation.from_messages(messages)

            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    break

                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)

                if max_tokens < self.cfg.buffer_tokens:
                    break

                stream = self.client.completions.create(
                    model=self.cfg.served_model_name,
                    temperature=self.cfg.temperature,
                    logprobs=self.cfg.top_logprobs,
                    max_tokens=max_tokens,
                    prompt=prompt_ids,
                    seed=attempt_seed,
                    stream=True,
                    extra_body={
                        'min_p': self.cfg.min_p,
                        'stop_token_ids': self.stop_token_ids,
                        'return_token_ids': True
                    }
                )

                try:
                    token_buffer = []
                    text_chunks = []

                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break

                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text

                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            total_tokens += len(new_tokens)
                            text_chunks.append(new_text)

                            chunk_logprobs = chunk.choices[0].logprobs

                            if chunk_logprobs is not None:
                                if chunk_logprobs.top_logprobs:
                                    logprobs_buffer.extend(chunk_logprobs.top_logprobs)

                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = self._scan_for_answer(search_text)

                            if answer is not None:
                                final_answer = answer
                                break

                finally:
                    stream.close()

                if final_answer is not None:
                    break

                if not token_buffer:
                    break

                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]

                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    final_answer = self._scan_for_answer(answer_text)
                    break

                if last_message.recipient == 'python':
                    python_calls += 1
                    tool_responses = local_tool.process_sync_plus(last_message)

                    response_text = tool_responses[0].content[0].text

                    if response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text:
                        python_errors += 1

                    conversation.messages.extend(tool_responses)

        except Exception as exc:
            python_errors += 1

        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)

        mean_entropy = self._compute_mean_entropy(logprobs_buffer)

        return {
            'Attempt': attempt_index + 1,
            'Response Length': total_tokens,
            'Python Calls': python_calls,
            'Python Errors': python_errors,
            'Entropy': mean_entropy,
            'Answer': final_answer
        }

    def _select_answer(self, detailed_results: list) -> int:

        answer_weights = defaultdict(float)
        answer_votes = defaultdict(int)

        for result in detailed_results:
            answer = result['Answer']
            entropy = result['Entropy']

            if answer is not None:
                weight = 1.0 / max(entropy, 1e-9)

                answer_weights[answer] += weight
                answer_votes[answer] += 1

        scored_answers = []

        for answer, total_weight in answer_weights.items():
            scored_answers.append({
                'answer': answer,
                'votes': answer_votes[answer],
                'score': total_weight
            })

        scored_answers.sort(key=lambda x: x['score'], reverse=True)

        vote_data = []

        for item in scored_answers:
            vote_data.append((
                item['answer'],
                item['votes'],
                item['score']
            ))

        vote_dataframe = pd.DataFrame(
            vote_data,
            columns=['Answer', 'Votes', 'Score']
        )

        vote_dataframe = vote_dataframe.round({'Score': 3})
        display(vote_dataframe)

        if not scored_answers:
            print('\nFinal Answer: 0\n')
            return 0

        final_answer = scored_answers[0]['answer']
        print(f'\nFinal Answer: {final_answer}\n')

        return final_answer

    def _compute_problem_timeout(self, time_left: float, remaining_problems: int) -> float:
        base = float(self.cfg.base_problem_timeout)   # 300
        high = float(self.cfg.high_problem_timeout)   # 900

        R = max(1, int(remaining_problems))
        T = max(0.0, float(time_left))

        budget = T - base * (R - 1)
        budget = min(budget, high)
        budget = max(budget, base)
        return budget


    def solve_problem(self, problem: str) -> int:
        # Fast similarity search over cert problems (MinHash-LSH on word shingles)
        example_hint = ""
        if self.cfg.use_rag:
            CERT_EXAMPLES_WITH_EMBEDDINGS = self.CERT_EXAMPLES_WITH_EMBEDDINGS


            if CERT_EXAMPLES_WITH_EMBEDDINGS:
                best_sim, best_cert = 0.0, None

                if problem is not None:
                    # Primary: LSH candidates + exact shingle-Jaccard on candidates
                    if getattr(self, "cert_lsh", None) is not None and getattr(self, "cert_by_id", None) is not None:
                        hits = self.cert_lsh.query(problem, top_k=1, min_candidates=50)
                        if hits:
                            best_sim, doc_id = hits[0]
                            best_cert = self.cert_by_id.get(doc_id)

                    # Fallback: brute-force token Jaccard (small corpora / if index unavailable)
                    if best_cert is None:
                        q_tokens = set(preprocess_text(problem).split())
                        for cert in CERT_EXAMPLES_WITH_EMBEDDINGS:
                            c_tokens = set((cert.get("problem_processed") or "").split())
                            if not q_tokens or not c_tokens:
                                continue
                            sim = len(q_tokens & c_tokens) / len(q_tokens | c_tokens)
                            if sim > best_sim:
                                best_sim, best_cert = sim, cert

                if best_sim >= 0.01 and best_cert is not None:
                    print("Best_sim = {} cert = {}".format(best_sim, best_cert))
                    parts = []
                    if best_cert.get("key_idea"):
                        parts.append(f"Key idea: {best_cert['key_idea']}")
                    if best_cert.get("proof_skeleton"):
                        skel = best_cert["proof_skeleton"]
                        parts.append("Proof skeleton: " + (" ".join(skel) if isinstance(skel, list) else skel))
                    if best_cert.get("attainment_or_example"):
                        parts.append(f"Attainment/example: {best_cert['attainment_or_example']}")
                    if best_cert.get("sanity_checks"):
                        ch = best_cert["sanity_checks"]
                        parts.append("Sanity checks: " + (" ".join(ch) if isinstance(ch, list) else ch))
                    fail = best_cert.get("failure") or {}
                    if isinstance(fail, dict) and (fail.get("minimal_fix") or fail.get("error_localization")):
                        parts.append(f"Common pitfall: {fail.get('error_localization', '')} Minimal fix: {fail.get('minimal_fix', '')}")
                    if parts:
                        example_hint = "Example problem can be used as hint:\n" + "\n".join(parts) + "\n\n"

        print(f'\nProblem: {problem}\n')

        user_input = f'{problem} {self.cfg.preference_prompt} {example_hint}'


        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global

        # remaining problems INCLUDING current
        remaining_problems = max(1, int(self.problems_remaining))

        budget = self._compute_problem_timeout(time_left=time_left, remaining_problems=remaining_problems)


        deadline = time.time() + budget

        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')

        tasks = []

        for attempt_index in range(self.cfg.attempts):
            tasks.append((self.cfg.system_prompt, attempt_index))

        detailed_results = []
        valid_answers = []

        stop_event = threading.Event()

        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)

        try:
            futures = []

            for (system_prompt, attempt_index) in tasks:
                future = executor.submit(
                    self._process_attempt,
                    user_input,
                    system_prompt,
                    attempt_index,
                    stop_event,
                    deadline
                )

                futures.append(future)

            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)

                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])

                    counts = Counter(valid_answers).most_common(1)

                    if counts and counts[0][1] >= self.cfg.early_stop:
                        stop_event.set()

                        for f in futures:
                            f.cancel()

                        break

                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue

        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)

            self.problems_remaining = max(0, self.problems_remaining - 1)

        if detailed_results:
            results_dataframe = pd.DataFrame(detailed_results)
            results_dataframe['Entropy'] = results_dataframe['Entropy'].round(3)
            results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64')

            display(results_dataframe)

        if not valid_answers:
            print('\nResult: 0\n')

            return 0

        return self._select_answer(detailed_results)

    def __del__(self):

        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()

        if hasattr(self, 'log_file'):
            self.log_file.close()

        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()

                except Exception:
                    pass

## 10. Initialize Solver

This starts the vLLM server, loads the model, and initializes Jupyter kernels. Takes 2-5 minutes depending on hardware.

In [87]:
try:
    solver = AIMO3Solver(CFG)
    print('SOLVER READY')
except Exception as e:
    print(f'SOLVER FAILED: {e}')
    import traceback; traceback.print_exc()
    try:
        with open('vllm_server.log') as f:
            print(f'vLLM logs:\n{f.read()[:5000]}')
    except: pass
    solver = None

Loading model weights from ./gpt-oss-120b into OS Page Cache...
Processed 46 files (65.28 GB) in 4.77 seconds.

Waiting for vLLM server...
Server is ready (took 60.98 seconds).

Initializing 16 persistent Jupyter kernels...
Kernels initialized in 1.86 seconds.

SOLVER READY


## 11. Run Inference

### Option A: Run on a CSV file
Upload a CSV with columns `id` and `problem`, then run the cell below.

In [ ]:
# === Option A: CSV input ===
# Upload your CSV or provide a path
import pandas as pd
INPUT_CSV = "hard_49_math_problems_set_v6.csv"   # Change this to your file path
OUTPUT_CSV = "answers.csv"
import time
if os.path.exists(INPUT_CSV) and solver is not None:
    import csv as csv_mod

    problems = []
    with open(INPUT_CSV, 'r') as f:
        reader = csv_mod.DictReader(f)
        for row in reader:
            problems.append((row['id'], row['problem']))

    solver.problems_remaining = len(problems)
    results = []

    for pid, problem_text in problems:
        gc.disable()
        try:
            answer = solver.solve_problem(problem_text)
            time.sleep(10)
        except Exception as e:
            print(f'Error on {pid}: {e}')
            answer = 0
        gc.enable()
        gc.collect()
        results.append({'id': pid, 'answer': answer})

    with open(OUTPUT_CSV, 'w', newline='') as f:
        writer = csv_mod.DictWriter(f, fieldnames=['id', 'answer'])
        writer.writeheader()
        writer.writerows(results)

    print(f'\nResults written to {OUTPUT_CSV}')
    print(f'Solved {len(results)} problems')
else:
    print(f"No CSV found at {INPUT_CSV} or solver not initialized. Use Option B below.")


Problem: Find the number of rectangles that can be formed inside a fixed regular dodecagon ($12$-gon) where each side of the rectangle lies on either a side or a diagonal of the dodecagon. The diagram below shows three of those rectangles.
[asy] unitsize(0.6 inch); for(int i=0; i<360; i+=30) { dot(dir(i), 4+black); draw(dir(i)--dir(i+30)); } draw(dir(120)--dir(330)); filldraw(dir(210)--dir(240)--dir(30)--dir(60)--cycle, mediumgray, linewidth(1.5)); draw((0,0.366)--(0.366,0), linewidth(1.5)); [/asy]

Budget: 900.00 seconds | Deadline: 1776767094.85



,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,4,17068,8,0,0.769,975
1,1,19999,18,1,0.868,315
2,6,20909,25,4,0.856,27
3,3,24846,33,0,0.775,12
4,2,28437,30,5,0.761,27
5,7,34316,45,6,0.595,315
6,5,36286,44,0,0.792,315
7,8,40191,45,0,0.797,315


,Answer,Votes,Score
0,315,4,5.351
1,27,2,2.481
2,975,1,1.301
3,12,1,1.290



Final Answer: 315


Problem: A semi-circle of radius 8 cm, rocks back and forth along a line.  The distance between the line on which the semi-circle sits and the line above is 12 cm.  As it rocks without slipping, the semi-circle touches the line above at two points.  (When the semi-circle hits the line above, it immediately rocks back in the other direction.)  What is the distance between these two points, in millimetres, rounded off to the nearest whole number? [asy]

draw((-15, -8)--(15, -8));draw((-15, 4)--(15, 4));draw((-8, 0)--(8, 0){down}..{up}(-8, 0));

[/asy] (Note: After finding the exact value of the desired distance, you may find a calculator useful to round this value off to the nearest whole number.)

Budget: 900.00 seconds | Deadline: 1776767533.52



,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,8,7082,1,0,0.988,139
1,6,11401,0,0,1.010,139
2,5,13537,1,0,0.940,139
3,4,14189,10,0,0.917,55
4,7,17863,8,0,0.887,55
5,2,19728,5,0,0.889,55
6,3,34237,5,0,0.869,139


...

,Answer,Votes,Score
0,2145,4,4.897
1,3283,1,1.141



Final Answer: 2145


Problem: Lines $m_{1}$, $m_{2}$, $l_{1}$ and $l_{2}$ are coplanar, and they are drawn such that $l_{1}$ is parallel to $l_{2}$, and $m_{2}$ is perpendicular to $l_{2}$. If the measure of angle 1 is 50 degrees, what is the measure in degrees of angle 2 in the figure below?

[asy]
draw((-6,0)--(3.5,0),Arrows);
draw((-4,3)--(2,-1.5),Arrows);
draw((3,3)--(-4.5,-4.5),Arrows);
draw((-4,1)--(2,-3.5),Arrows);
label("1",(1.5,0),N);
label("2",(-2.7,0),N);
label("$m_{2}$",(3,3),NE);
label("$m_{1}$",(3.5,0),E);
label("$l_{1}$",(2,-1.5),E);
label("$l_{2}$",(2,-3.5),E);
[/asy]

Budget: 900.00 seconds | Deadline: 1776768451.07



,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,7,3623,2,0,0.803,40
1,3,6344,0,0,0.828,40
2,8,7401,0,0,0.768,40
3,2,7692,1,0,0.785,40


,Answer,Votes,Score
0,40,4,5.027



Final Answer: 40


Problem: Consider two points $A(-1, 1)$ and $B(1, -1)$ on the coordinate plane. Let $P$ be a point such that the absolute value of the $x$-coordinate of $P$ is less than or equal to 1. Find the area of the domain of points $P$ satisfying the following conditions:

(i) There exists a parabola with the absolute value of the $x$-coordinate of the vertex greater than or equal to 1 that passes through points $A$, $P$, and $B$.

(ii) The points $A$, $P$, and $B$ are collinear.

Budget: 900.00 seconds | Deadline: 1776768540.24



### Option B: Run on a single problem

In [ ]:
# === Option B: Single problem ===
if solver is not None:
    problem = "Find the remainder when 2^2025 is divided by 17."

    answer = solver.solve_problem(problem)
    print(f"\n{'='*60}")
    print(f"ANSWER: {answer}")
    print(f"{'='*60}")

## 12. Cleanup

In [ ]:
if solver is not None:
    del solver
    print("Solver shut down")